# 📓 [Day 30 실전 워크북] Cypher 심화: 다차원 경로 탐색 & WITH 파이프라인 핸즈온

> **학습 목표**:
> 1. 가변 길이 경로(`*1..2`, `*0..2`)와 0-Hop의 본질을 이해하고 순회한다.
> 2. `shortestPath` 및 `allShortestPaths`로 최단 경로를 찾고 물리적 거리/시간을 합산한다.
> 3. `nodes(p)`, `relationships(p)`, 리스트 컴프리헨션 및 `all`/`any`/`none` 술어로 경로를 해부한다.
> 4. 패턴 술어(`WHERE NOT ()`, `EXISTS { }`)로 관계 유무를 필터링한다.
> 5. `OPTIONAL MATCH`의 NULL 함정을 `WITH` 격리 파이프라인으로 해결하고, 정렬/페이징(`SKIP`/`LIMIT`)을 마스터한다.

---

In [ ]:
# [환경 설정] Neo4j 연결 및 헬퍼 함수 정의
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)
load_dotenv("내작업폴더/day28_Neo4j_설치_Movies/.env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "test0011")
AURA_URI = os.getenv("AURA_URI")
AURA_USER = os.getenv("AURA_USER")
AURA_PASSWORD = os.getenv("AURA_PASSWORD")

driver = None
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✅ 로컬 Neo4j 연결 성공:", NEO4J_URI)
except Exception:
    if AURA_URI and AURA_USER and AURA_PASSWORD:
        driver = GraphDatabase.driver(AURA_URI, auth=(AURA_USER, AURA_PASSWORD))
        driver.verify_connectivity()
        print("✅ Neo4j Aura 클라우드 연결 성공:", AURA_URI)
    else:
        raise ConnectionError("Neo4j에 연결할 수 없습니다. .env를 확인하세요.")

def run_cypher(query: str, **params):
    """Cypher 쿼리 실행 후 dict 리스트로 반환하는 공용 헬퍼"""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

In [ ]:
# [초기화 및 시드 적재] 실습 격리 네임스페이스 (SmartHub, SmartCity, SmartSpot 등)
run_cypher("MATCH (n:SmartHub) DETACH DELETE n")
run_cypher("MATCH (n:SmartCity) DETACH DELETE n")
run_cypher("MATCH (n:SmartSpot) DETACH DELETE n")
run_cypher("MATCH (n:SmartManager) DETACH DELETE n")

# 1) 스마트 물류망 시드
run_cypher("""
CREATE (h1:SmartHub {name: '인천메가허브', hub_id: 'H01', capacity: 50000}),
       (h2:SmartHub {name: '군포허브',     hub_id: 'H02', capacity: 30000}),
       (h3:SmartHub {name: '대전허브',     hub_id: 'H03', capacity: 40000}),
       (h4:SmartHub {name: '대구허브',     hub_id: 'H04', capacity: 25000}),
       (c1:SmartCity {name: '서울강남',   city_id: 'C01', zone: '수도권'}),
       (c2:SmartCity {name: '수원',       city_id: 'C02', zone: '경기남부'}),
       (c3:SmartCity {name: '천안',       city_id: 'C03', zone: '충청권'}),
       (c4:SmartCity {name: '부산',       city_id: 'C04', zone: '영남권'}),
       (c5:SmartCity {name: '제주',       city_id: 'C05', zone: '도서산간'})

CREATE (h1)-[:TRUCK_ROUTE {time: 45,  cost: 15000, distance_km: 35}]->(c1),
       (h1)-[:TRUCK_ROUTE {time: 50,  cost: 18000, distance_km: 42}]->(h2),
       (h2)-[:TRUCK_ROUTE {time: 30,  cost: 12000, distance_km: 25}]->(c2),
       (h2)-[:TRUCK_ROUTE {time: 80,  cost: 25000, distance_km: 85}]->(h3),
       (h3)-[:TRUCK_ROUTE {time: 40,  cost: 14000, distance_km: 38}]->(c3),
       (h3)-[:TRUCK_ROUTE {time: 90,  cost: 30000, distance_km: 120}]->(h4),
       (h4)-[:TRUCK_ROUTE {time: 60,  cost: 20000, distance_km: 75}]->(c4),
       (h1)-[:AIR_ROUTE   {time: 120, cost: 80000, distance_km: 450}]->(c5),
       (h1)-[:AIR_ROUTE   {time: 60,  cost: 50000, distance_km: 380}]->(c4)
""")

# 2) 다이닝 & 캠핑 스팟 시드
run_cypher("""
CREATE (s1:SmartSpot {name: '포레스트 글램핑', category: '글램핑', area: '가평', price: 180000, rating: 4.8}),
       (s2:SmartSpot {name: '별빛 오토캠핑장', category: '오토캠핑', area: '가평', price: 60000,  rating: 4.5}),
       (s3:SmartSpot {name: '오션뷰 카라반',   category: '카라반', area: '강릉', price: 150000, rating: 4.9}),
       (s4:SmartSpot {name: '마운틴 힐링파크', category: '오토캠핑', area: '평창', price: 50000,  rating: 4.2}),
       (s5:SmartSpot {name: '도심속 루프탑가든', category: '글램핑', area: '서울', price: 220000, rating: 4.6}),
       (m1:SmartManager {name: '김총괄', phone: '010-1111-2222', grade: 'Master'}),
       (m2:SmartManager {name: '이관리', phone: '010-3333-4444', grade: 'Senior'})

CREATE (s1)-[:MANAGED_BY]->(m1),
       (s2)-[:MANAGED_BY]->(m1),
       (s3)-[:MANAGED_BY]->(m2)
""")
print("🚀 [초기화 및 시드 적재 완료!]")

## 🎯 미션 1. 가변 길이 패턴 순회 (*1..2)
- `인천메가허브`에서 육상 노선(`TRUCK_ROUTE`)을 따라 **1~2단계** 안에 도달할 수 있는 모든 도착 지점의 이름(`dest_name`)을 중복 없이(`DISTINCT`) 오름차순으로 반환하세요.

In [ ]:
# [TODO] 미션 1 쿼리 작성
q1 = """
MATCH (start:SmartHub {name: '인천메가허브'})-[:TRUCK_ROUTE*1..2]->(dest)
RETURN DISTINCT dest.name AS dest_name
ORDER BY dest_name ASC
"""
res1 = run_cypher(q1)
print("미션 1 결과:", res1)

In [ ]:
# [자가채점] 미션 1 검증
expected = ['군포허브', '대전허브', '서울강남', '수원']
assert [r['dest_name'] for r in res1] == expected, f"Expected {expected}, got {res1}"
print("✅ [미션 1 통과!] 가변 길이 경로 순회가 정상 작동합니다.")

## 🎯 미션 2. shortestPath 최단 경로 & length 계산
- `인천메가허브`에서 `부산`까지 노선 종류(`TRUCK_ROUTE` 또는 `AIR_ROUTE`)에 상관없이 홉 수 기준 최단 경로 `p`를 구하고, 경유 노드 리스트(`node_names`), 경로 홉 수(`hop_count`)를 반환하세요.

In [ ]:
# [TODO] 미션 2 쿼리 작성
q2 = """
MATCH p = shortestPath((start:SmartHub {name: '인천메가허브'})-[:TRUCK_ROUTE|AIR_ROUTE*]-(dest:SmartCity {name: '부산'}))
RETURN [n IN nodes(p) | n.name] AS node_names, length(p) AS hop_count
"""
res2 = run_cypher(q2)
print("미션 2 결과:", res2)

In [ ]:
# [자가채점] 미션 2 검증
assert len(res2) == 1
assert res2[0]['hop_count'] == 1 and res2[0]['node_names'] == ['인천메가허브', '부산']
print("✅ [미션 2 통과!] shortestPath 탐색이 정상 작동합니다.")

## 🎯 미션 3. 리스트 술어 함수 (all)
- `인천메가허브`에서 육상 노선(`TRUCK_ROUTE`)을 타고 도달 가능한 모든 도시(`SmartCity`) 중, **모든 이동 구간의 소요시간(time)이 60분 이하**(`all`)인 도시 이름(`city_name`)을 반환하세요.

In [ ]:
# [TODO] 미션 3 쿼리 작성
q3 = """
MATCH p = (start:SmartHub {name: '인천메가허브'})-[:TRUCK_ROUTE*]->(c:SmartCity)
WHERE all(r IN relationships(p) WHERE r.time <= 60)
RETURN DISTINCT c.name AS city_name
ORDER BY city_name
"""
res3 = run_cypher(q3)
print("미션 3 결과:", res3)

In [ ]:
# [자가채점] 미션 3 검증
assert [r['city_name'] for r in res3] == ['서울강남', '수원']
print("✅ [미션 3 통과!] all() 리스트 술어가 정상 작동합니다.")

## 🎯 미션 4. 패턴 술어 (WHERE NOT () & EXISTS { })
- 관리자(`SmartManager`)가 연결되지 않은 스팟(`SmartSpot`)의 이름(`spot_name`)을 오름차순으로 반환하세요.

In [ ]:
# [TODO] 미션 4 쿼리 작성
q4 = """
MATCH (s:SmartSpot)
WHERE NOT (s)-[:MANAGED_BY]->(:SmartManager)
RETURN s.name AS spot_name
ORDER BY spot_name
"""
res4 = run_cypher(q4)
print("미션 4 결과:", res4)

In [ ]:
# [자가채점] 미션 4 검증
expected_unmanaged = ['도심속 루프탑가든', '마운틴 힐링파크']
assert [r['spot_name'] for r in res4] == expected_unmanaged
print("✅ [미션 4 통과!] 패턴 부재(NOT) 술어가 정상 작동합니다.")

## 🎯 미션 5. OPTIONAL MATCH + WITH 파이프라인 & 페이징
- 모든 스팟(`SmartSpot`)에 대해 `OPTIONAL MATCH`로 관리자를 조회하고, `WITH` 파이프라인으로 1인당 예상 비용(`price / 2 AS unit_price`)을 계산한 뒤, 가격(`price`) 오름차순 및 이름(`name`) 오름차순으로 정렬하여 **2페이지(1페이지당 2건: SKIP 2 LIMIT 2)**를 조회하세요.

In [ ]:
# [TODO] 미션 5 쿼리 작성
q5 = """
MATCH (s:SmartSpot)
OPTIONAL MATCH (s)-[:MANAGED_BY]->(m:SmartManager)
WITH s, coalesce(m.name, '미배정') AS manager, (s.price / 2) AS unit_price
ORDER BY s.price ASC, s.name ASC
SKIP 2
LIMIT 2
RETURN s.name AS name, s.price AS price, unit_price, manager
"""
res5 = run_cypher(q5)
print("미션 5 결과:", res5)

In [ ]:
# [자가채점] 미션 5 검증
assert len(res5) == 2
assert res5[0]['name'] == '오션뷰 카라반' and res5[0]['manager'] == '이관리'
assert res5[1]['name'] == '포레스트 글램핑' and res5[1]['manager'] == '김총괄'
print("✅ [미션 5 통과!] OPTIONAL MATCH와 WITH 페이징이 완벽하게 작동합니다.")